# Whole Foods Location Similarity Model

This notebook develops a similarity-based model to identify census tracts that
resemble neighborhoods where Whole Foods currently operates.

Using tract-level socioeconomic and retail characteristics, the model compares
each census tract to the profile of existing Whole Foods locations and ranks
tracts by similarity in order to identify potential candidate locations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Step 1. Set up Spark and project paths

In this section, we initialize a Spark session and define the project paths.

The four input files are tract-level tables that were already created in the preprocessing stage:
- `tract_acs.csv`
- `tract_crime.csv`
- `tract_osm.csv`
- `tract_wf.csv`

These files are stored in the `processed_data` folder in Google Drive.

Our goal in this step is to:
1. Read all four tract-level datasets with Spark
2. Standardize the join key (`GEOID`)
3. Merge them into one master tract-level table

This master table will be the foundation for the later ranking pipeline.

In [ ]:
# Step 1: Initialize Spark and define paths

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Create Spark session
spark = SparkSession.builder \
    .appName("WholeFoodsLocationAnalysis") \
    .getOrCreate()

# Define the base project path in Google Drive
BASE_PATH = "/content/drive/MyDrive/msba405-group-project/"
PROCESSED_PATH = BASE_PATH + "processed_data/"

# Define full file paths
ACS_PATH = PROCESSED_PATH + "tract_acs.csv"
CRIME_PATH = PROCESSED_PATH + "tract_crime.csv"
OSM_PATH = PROCESSED_PATH + "tract_osm.csv"
WF_PATH = PROCESSED_PATH + "tract_wf.csv"

# Step 2. Read the processed tract-level datasets

We use Spark to read each CSV file as a Spark DataFrame.

Important settings:
- `header=True`: use the first row as column names
- `inferSchema=True`: let Spark infer data types automatically

At this stage, we are not modifying the data yet.  
We are only loading the four tables into Spark so that we can inspect and merge them.

In [ ]:
# Step 2: Read the four processed CSV files

# Read ACS tract-level data
acs = spark.read.csv(
    ACS_PATH,
    header=True,
    inferSchema=True
)

# Read crime tract-level data
crime = spark.read.csv(
    CRIME_PATH,
    header=True,
    inferSchema=True
)

# Read OSM tract-level data
osm = spark.read.csv(
    OSM_PATH,
    header=True,
    inferSchema=True
)

# Read Whole Foods tract-level data
wf = spark.read.csv(
    WF_PATH,
    header=True,
    inferSchema=True
)

## Step 3. Inspect schemas and key columns

Before merging the four tract-level datasets, we inspect their:
- row counts
- column names
- data types
- key column (`GEOID`)

This is an important quality-control step because merge problems often come from:
- inconsistent join keys
- unexpected data types
- duplicate tract IDs
- missing or ambiguous columns


In [ ]:
# Step 3: Inspect the datasets

# Print row counts
print("ACS row count:", acs.count())
print("Crime row count:", crime.count())
print("OSM row count:", osm.count())
print("WF row count:", wf.count())

# Print column names
print("ACS columns:", acs.columns)
print("Crime columns:", crime.columns)
print("OSM columns:", osm.columns)
print("WF columns:", wf.columns)

# Print schemas
print("ACS schema:")
acs.printSchema()

print("Crime schema:")
crime.printSchema()

print("OSM schema:")
osm.printSchema()

print("WF schema:")
wf.printSchema()

# Preview the GEOID column in each dataset
acs.select("GEOID").show(5, truncate=False)
crime.select("GEOID").show(5, truncate=False)
osm.select("GEOID").show(5, truncate=False)
wf.select("GEOID").show(5, truncate=False)

# Step 4. Standardize the join key and prepare for merge

To merge safely, all datasets must use the same join key format.

We standardize `GEOID` by:
1. Casting it to string in every table
2. Keeping one tract per row
3. Preparing each dataset for a tract-level left join

We will use ACS as the base table because it usually contains the core demographic features for every tract.

In [ ]:
# Step 4: Standardize GEOID type

# Cast GEOID to string in every dataset
acs = acs.withColumn("GEOID", col("GEOID").cast("string"))
crime = crime.withColumn("GEOID", col("GEOID").cast("string"))
osm = osm.withColumn("GEOID", col("GEOID").cast("string"))
wf = wf.withColumn("GEOID", col("GEOID").cast("string"))

# Step 5. Check for duplicate tract keys before merging

A clean tract-level master table should have one row per `GEOID`.

Before joining, we check whether any dataset contains duplicate tract IDs.
If duplicates exist, we should resolve them before merging. Otherwise, the join may create unexpected row expansion.

In [ ]:
# Step 5: Check duplicate GEOID values

from pyspark.sql.functions import count

# Check duplicate GEOIDs in ACS
acs.groupBy("GEOID").count().filter(col("count") > 1).show()

# Check duplicate GEOIDs in crime
crime.groupBy("GEOID").count().filter(col("count") > 1).show()

# Check duplicate GEOIDs in OSM
osm.groupBy("GEOID").count().filter(col("count") > 1).show()

# Check duplicate GEOIDs in WF
wf.groupBy("GEOID").count().filter(col("count") > 1).show()

## Step 6. Merge tract-level datasets

After validating the schemas and ensuring that GEOID is consistent
across all datasets, we merge the four tract-level tables into a
single master dataset.

Merge strategy:

- Use **ACS as the base table** because it provides the full set
  of census tracts and core demographic variables.
- Join other datasets using **left joins on GEOID**.
- This preserves all tracts while attaching crime, retail, and
  Whole Foods indicators where available.

The resulting table will serve as the **master tract-level dataset**
for the location scoring pipeline.

In [ ]:
# Step 6: Merge tract-level datasets

# Start with ACS as the base table
master_df = acs

# Join crime dataset
master_df = master_df.join(
    crime,
    on="GEOID",
    how="left"
)

# Join OSM dataset
master_df = master_df.join(
    osm,
    on="GEOID",
    how="left"
)

# Join Whole Foods dataset
master_df = master_df.join(
    wf,
    on="GEOID",
    how="left"
)

## Step 7. Validate the merged master dataset

After performing the joins, we validate the merged table by checking:

- row count
- schema structure
- sample rows
- uniqueness of GEOID

This ensures that the join did not introduce duplicated rows or
unexpected schema conflicts.

In [ ]:
# Step 7: Validate merged dataset

# Check row count
print("Master table row count:", master_df.count())

# Print columns
print("\nMaster table columns:")
print(master_df.columns)

# Print schema
print("\nMaster table schema:")
master_df.printSchema()

# Preview rows
master_df.show(5, truncate=False)

# Check duplicate GEOIDs after merge
master_df.groupBy("GEOID").count().filter(col("count") > 1).show()

## Step 8. Select modeling features

From the merged master dataset, we select the variables required
for the location similarity model.

Key features:

- median household income
- bachelor degree rate
- population density
- retail density

Additional variables used for filtering:

- crime indicator
- existing Whole Foods indicator

In [ ]:
# Step 8: Select modeling features

features_df = master_df.select(
    "GEOID",
    "income",
    "ba_rate",
    "pop_density",
    "retail_density",
    "crime_low_60",
    "has_wf"
)

features_df.show(5)

## Step 9. Filter candidate tracts

Before computing similarity scores, we define the set of candidate tracts.

Filtering rules:

1. Remove tracts with high crime levels.
   We keep only tracts where `crime_low_60 = true`.

2. Remove tracts that already contain a Whole Foods store.
   These tracts are not candidates for new locations.

The remaining tracts will be used for similarity scoring.

In [ ]:
# Step 9: Filter candidate tracts

candidate_df = features_df.filter(
    (col("crime_low_60") == True) &
    (col("has_wf") == 0)
)

# Remove rows with missing feature values
candidate_df = candidate_df.dropna(
    subset=["income", "ba_rate", "pop_density", "retail_density"]
)

print("Candidate tract count:", candidate_df.count())

candidate_df.show(5)

## Step 10. Compute standardization statistics

Before measuring similarity, we standardize the modeling variables.

This is necessary because the features are measured on very different scales:
- income is measured in tens of thousands
- population density can be in the thousands
- retail density is much smaller

To make the variables comparable, we compute the mean and standard deviation
for each feature and transform them into z-scores.

In [ ]:
# Step 10: Compute mean and standard deviation from the full feature table

from pyspark.sql.functions import mean, stddev

stats = features_df.select(
    mean("income").alias("income_mean"),
    stddev("income").alias("income_std"),
    mean("ba_rate").alias("ba_mean"),
    stddev("ba_rate").alias("ba_std"),
    mean("pop_density").alias("pop_mean"),
    stddev("pop_density").alias("pop_std"),
    mean("retail_density").alias("retail_mean"),
    stddev("retail_density").alias("retail_std")
).collect()[0]

## Step 11. Standardize modeling features

Using the summary statistics from the previous step, we convert each feature
into a z-score.

This puts all variables on the same scale so that no single variable
dominates the distance calculation.

In [ ]:
# Step 11: Standardize the full feature table

from pyspark.sql.functions import col

features_z_df = features_df.withColumn(
    "z_income",
    (col("income") - stats["income_mean"]) / stats["income_std"]
).withColumn(
    "z_ba_rate",
    (col("ba_rate") - stats["ba_mean"]) / stats["ba_std"]
).withColumn(
    "z_pop_density",
    (col("pop_density") - stats["pop_mean"]) / stats["pop_std"]
).withColumn(
    "z_retail_density",
    (col("retail_density") - stats["retail_mean"]) / stats["retail_std"]
)

# Rebuild candidate set in standardized space
candidate_z_df = features_z_df.filter(
    (col("crime_low_60") == True) &
    (col("has_wf") == 0)
).dropna(
    subset=["income", "ba_rate", "pop_density", "retail_density"]
)

candidate_z_df.show(5)

## Step 12. Compute the standardized Whole Foods centroid

Next, we compute the average z-score values for tracts that already contain
a Whole Foods store.

This gives us the standardized Whole Foods profile, which serves as the
reference point for similarity scoring.

In [ ]:
# Step 12: Compute Whole Foods centroid in standardized space

wf_z_profile = features_z_df.filter(
    col("has_wf") == 1
).select(
    "z_income",
    "z_ba_rate",
    "z_pop_density",
    "z_retail_density"
).groupBy().avg()

wf_z_stats = wf_z_profile.collect()[0]

wf_z_income = wf_z_stats["avg(z_income)"]
wf_z_ba = wf_z_stats["avg(z_ba_rate)"]
wf_z_pop = wf_z_stats["avg(z_pop_density)"]
wf_z_retail = wf_z_stats["avg(z_retail_density)"]

wf_z_profile.show()

## Step 13. Compute Euclidean distance in standardized space

We now measure the Euclidean distance between each candidate tract
and the standardized Whole Foods centroid.

Smaller distances indicate tracts that are more similar to existing
Whole Foods neighborhoods.

In [ ]:
from pyspark.sql.functions import sqrt

scored_df = candidate_z_df.withColumn(
    "distance",
    sqrt(
        (col("z_income") - wf_z_income) ** 2 +
        (col("z_ba_rate") - wf_z_ba) ** 2 +
        (col("z_pop_density") - wf_z_pop) ** 2 +
        (col("z_retail_density") - wf_z_retail) ** 2
    )
)

scored_df.select(
    "GEOID",
    "z_income",
    "z_ba_rate",
    "z_pop_density",
    "z_retail_density",
    "distance"
).show(5)

## Step 14. Convert distance to similarity score

To make the ranking more intuitive, we convert the Euclidean distance into a similarity score.

The similarity score is defined as:

$$
\textbf{score}_i = \frac{1}{1 + d_i}
$$

where:

- $d_i$ is the Euclidean distance between tract $i$ and the Whole Foods profile.

With this transformation:

- **Higher score → more similar to Whole Foods neighborhoods**
- **Lower score → less similar**

In [ ]:
scored_df = scored_df.withColumn(
    "score",
    1 / (1 + col("distance"))
)

## Step 15. Rank Top Candidate Locations

To highlight the most promising locations for potential Whole Foods expansion,
we present the **top 10 census tracts ranked by similarity score**.

Higher scores indicate tracts whose demographic and retail characteristics
are most similar to neighborhoods where Whole Foods currently operates.

In [ ]:
ranked_df = scored_df.orderBy(col("score").desc())

ranked_df.select(
    "GEOID",
    "income",
    "ba_rate",
    "pop_density",
    "retail_density",
    "distance",
    "score"
).show(10)

For visualization and dashboard presentation, numeric values are rounded
to improve readability. The rounding only affects display values and does
not change the underlying analytical results.

In [ ]:
from pyspark.sql.functions import round

dashboard_df = ranked_df.select(
    "GEOID",
    round("income",0).alias("income"),
    round("ba_rate",3).alias("ba_rate"),
    round("pop_density",1).alias("pop_density"),
    round("retail_density",2).alias("retail_density"),
    round("distance",3).alias("distance"),
    round("score",3).alias("score")
)

dashboard_df.show(10)

## Model Validation

To validate the similarity-based ranking model, we test whether census tracts
that already contain Whole Foods stores receive higher similarity scores
than other tracts.

This validation is performed on **all tracts**, including both:
- tracts with existing Whole Foods stores (`has_wf = 1`)
- tracts without Whole Foods stores (`has_wf = 0`)

If the model captures meaningful characteristics of Whole Foods neighborhoods,
tracts with existing Whole Foods stores should have **higher similarity scores**.

We evaluate the model using three checks:

1. **Average similarity score comparison**
2. **Median similarity score comparison**
3. **Recovery of existing Whole Foods tracts among the highest-scoring tracts**

In [ ]:
# Validation A: Score all tracts

from pyspark.sql.functions import col, sqrt, avg, round, expr

# Score all tracts in standardized feature space
all_scored_df = features_z_df.withColumn(
    "distance",
    sqrt(
        (col("z_income") - wf_z_income) ** 2 +
        (col("z_ba_rate") - wf_z_ba) ** 2 +
        (col("z_pop_density") - wf_z_pop) ** 2 +
        (col("z_retail_density") - wf_z_retail) ** 2
    )
)

all_scored_df = all_scored_df.withColumn(
    "score",
    1 / (1 + col("distance"))
)

In [ ]:
# Validation A1: Average score comparison

print("Validation A1: Average similarity score by Whole Foods presence")

avg_validation_df = all_scored_df.groupBy("has_wf").agg(
    round(avg("score"), 3).alias("avg_score")
)

avg_validation_df.show()

In [ ]:
# Validation A2: Median score comparison

print("Validation A2: Median and average similarity score by Whole Foods presence")

median_validation_df = all_scored_df.groupBy("has_wf").agg(
    round(expr("percentile_approx(score, 0.5)"), 3).alias("median_score"),
    round(avg("score"), 3).alias("avg_score")
)

median_validation_df.show()


In [ ]:
# Validation A3: Check whether existing WF tracts appear among the highest-scoring tracts

top_n_all = 50

print(f"Validation A3: Existing Whole Foods tracts in Top {top_n_all} of all tracts")

topn_all_df = all_scored_df.orderBy(col("score").desc()).limit(top_n_all)

topn_all_df.select(
    "GEOID",
    "has_wf",
    round("score", 3).alias("score"),
    round("distance", 3).alias("distance")
).show(20, truncate=False)

wf_in_topn_all = topn_all_df.filter(col("has_wf") == 1).count()
total_wf_tracts = all_scored_df.filter(col("has_wf") == 1).count()

recovery_rate = wf_in_topn_all / total_wf_tracts
share_top_n = wf_in_topn_all / top_n_all

print(f"Existing Whole Foods tracts in Top {top_n_all}: {wf_in_topn_all}")
print("Total existing Whole Foods tracts:", total_wf_tracts)
print("Recovery rate:", __builtins__.round(recovery_rate, 3))
print("Share of Top N:", __builtins__.round(share_top_n, 3))

### Validation Results

The validation results suggest that the similarity-based model captures
meaningful characteristics of Whole Foods locations. Tracts that already
contain Whole Foods stores have higher similarity scores, with a median score
of **0.402** compared to **0.286** for other tracts (average: **0.401 vs 0.304**).

Among the top 50 scoring tracts in the dataset, **2 already contain Whole Foods
stores** out of **28 existing Whole Foods tracts**, corresponding to a recovery
rate of **0.071**.

Although the recovery rate is modest, this is expected given the limited set of
tract-level variables used in the model. Overall, the results indicate that the
model captures part of the Whole Foods neighborhood profile and provides a
reasonable basis for identifying candidate locations.